In [ ]:
# STEP 1: Clone the project from GitHub into the Colab VM's local disk.
# No Google Drive mount needed — models are already trained and committed
# to the repo, so nothing here needs to persist between sessions. Re-running
# this cell in a later session just re-clones fresh (fast, no auth popups).
GITHUB_REPO_URL = "https://github.com/rayyantaimoor1/Ecommerce-customer-intelligence.git"
PROJECT_DIR = "/content/Ecommerce-customer-intelligence"

import os

if os.path.exists(f"{PROJECT_DIR}/.git"):
    print("Repo already exists — pulling latest changes...")
    os.chdir(PROJECT_DIR)
    os.system("git pull")
else:
    print(f"Cloning {GITHUB_REPO_URL} ...")
    os.system(f'git clone "{GITHUB_REPO_URL}" "{PROJECT_DIR}"')
    os.chdir(PROJECT_DIR)

print("\nWorking directory:", os.getcwd())
os.system("ls")

In [ ]:
# STEP 2: Install packages Colab doesn't include by default (FastAPI, Streamlit, etc).
!pip install -q fastapi uvicorn pydantic streamlit plotly nbformat nbconvert

print("Dependencies installed.")

In [ ]:
# STEP 3: Confirm the bundled dataset is present at data/raw/OnlineRetail.csv.
raw_path = "data/raw/OnlineRetail.csv"

if os.path.exists(raw_path):
    size_mb = os.path.getsize(raw_path) / (1024 * 1024)
    print(f"Found dataset at {raw_path} ({size_mb:.1f} MB) — ready to go.")
else:
    print(f"No file found at {raw_path}. Check that Step 1's clone succeeded, "
          f"or upload a file manually using the cell below.")

In [ ]:
# OPTIONAL: run this only to replace the bundled CSV with the real Kaggle/UCI dataset.

# from google.colab import files
# print("Choose your real OnlineRetail.csv (or Online Retail.xlsx exported as csv)...")
# uploaded = files.upload()
# real_file = list(uploaded.keys())[0]
# import shutil
# shutil.move(real_file, "data/raw/OnlineRetail.csv")
# print("Replaced data/raw/OnlineRetail.csv with your uploaded file.")

In [ ]:
# STEP 4: (Optional) Re-run all 6 pipeline notebooks in order — only needed if
# you swapped in a different dataset above or want to retrain from scratch.
# Skip this step entirely if you just want to use the pre-trained model
# already committed in models/clustering_pipeline.joblib.
notebooks = [
    "01_eda.ipynb",
    "02_rfm_feature_engineering.ipynb",
    "03_pca_tsne.ipynb",
    "04_kmeans_hierarchical.ipynb",
    "05_dbscan_anomaly.ipynb",
    "06_model_export.ipynb",
]

for nb in notebooks:
    print(f"Running notebooks/{nb} ...")
    result = os.system(
        f'jupyter nbconvert --to notebook --execute --inplace "notebooks/{nb}"'
    )
    status = "OK" if result == 0 else f"FAILED (exit code {result})"
    print(f"  {nb}: {status}")

print("\nAll notebooks finished. Check output above for any FAILED steps before continuing.")

In [ ]:
# STEP 5: Verify the trained pipeline (clustering_pipeline.joblib) loads correctly.
import joblib

bundle = joblib.load("models/clustering_pipeline.joblib")
print("Feature columns:", bundle["feature_columns"])
print("Number of clusters (K):", bundle["kmeans_model"].n_clusters)
print("Personas:")
for cid, name in sorted(bundle["persona_labels"].items()):
    print(f"  Cluster {cid}: {name}")

In [ ]:
# STEP 6a: Start Streamlit in the background inside the Colab VM (port 8501).
!pkill -f streamlit || true

os.chdir(f"{PROJECT_DIR}/dashboard")
get_ipython().system_raw(
    'streamlit run Home.py --server.port 8501 --server.headless true &>/content/logs.txt &'
)

print("Streamlit starting... waiting a few seconds for it to come up.")
import time
time.sleep(8)
print("Done. Run the next cell to get your public link.")

In [ ]:
# STEP 6b: Print the tunnel password (Colab VM's public IP) - paste this into the loca.lt page.
!wget -q -O - https://loca.lt/mytunnelpassword

In [ ]:
# STEP 6c: Start the public tunnel (keep this cell running; open the printed URL in your browser).
# Copy the printed URL into your browser, then paste the IP above into the
# "Tunnel Password" field on the page that loads.
!npx localtunnel --port 8501

In [ ]:
# TROUBLESHOOTING NOTES:
# - "fatal: repository not found" in Step 1 -> check GITHUB_REPO_URL is
#   correct and the repo is Public.
# - "npx: command not found" -> run: !apt-get install -y nodejs npm
#   in a new cell, then retry Step 6c.
# - Tunnel URL loads blank/error page -> Streamlit may still be starting;
#   re-run Step 6a, wait 10-15 seconds, then re-run Step 6c.
# - "Model pipeline not found" -> Step 4 didn't run/finish, or wasn't
#   needed since the pre-trained model is already in the repo. Re-check
#   Step 3's dataset path and re-run Step 4 if you changed the dataset.
# - Session disconnected -> just re-run from Step 1; it's a fresh clone
#   each time (fast) since nothing needs to persist in Drive.
# - Want the FastAPI docs UI instead of Streamlit? Run:
#   get_ipython().system_raw('cd ../app && uvicorn main:app --host 0.0.0.0 --port 8000 &>/content/api_logs.txt &')
#   then: !npx localtunnel --port 8000
#   and open <tunnel-url>/docs